# MiniMax H3 + ComfyUI（Colab 修正完全版）

- **trycloudflare.com 不使用**（Access Denied 対策）
- UI は **Colab 公式プロキシ**（`googleusercontent.com`）で開く
- ComfyUI **0.30.0+** / モデル: [Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3)

| 項目 | 推奨 |
|------|------|
| GPU | L4 / A100（無料 T4 は遅い） |
| ディスク | モデル約 **42.5 GB** |
| 起動後 | セル5が印刷する URL だけ開く |

**ランタイム → ランタイムのタイプを変更 → GPU** のあと、上から実行。
重みは Drive の `minimax-h3-comfyui/models`（フォルダ `1zd4IaS-pw1RJrx4TaGDGgS1ZpBsjeVRK`）を繋ぐ。Hugging Face からは落とさない。動画 VAE はそこにある fp16 を使う。

In [ ]:
#@title 0) GPU / ディスク確認
!nvidia-smi -L
!df -h /content | tail -1
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(props.total_memory / 1024**3, 1))


In [ ]:
#@title 1) Google Drive の models をマウント
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3-comfyui"  #@param {type:"string"}

if not USE_DRIVE:
    raise SystemExit("重みは Drive の models を使う。USE_DRIVE を True にする。")
from google.colab import drive
import os
print("Drive の接続画面では ys73497349@gmail.com を選ぶ。")
drive.mount("/content/drive")
models = os.path.join(DRIVE_ROOT, "models")
if not os.path.isdir(models):
    raise SystemExit("Drive の models が無い: " + models)
print("models:", models)

In [ ]:
#@title 2) ComfyUI clone + 依存関係（0.30.0 以降）
import os

COMFY_DIR = "/content/ComfyUI"
REPO = "https://github.com/Comfy-Org/ComfyUI.git"

if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 {REPO} {COMFY_DIR}
else:
    %cd {COMFY_DIR}
    !git pull --ff-only || true

%cd {COMFY_DIR}
!pip install -q -r requirements.txt

print("ComfyUI ready:", COMFY_DIR)
!git log -1 --oneline


In [ ]:
#@title 3) Drive の models を ComfyUI に繋ぐ（ダウンロードしない）
import importlib.util
import os
import subprocess
from pathlib import Path

COMFY_DIR = "/content/ComfyUI"
DRIVE_MODELS = "/content/drive/MyDrive/minimax-h3-comfyui/models"
MODE = "r2v"  #@param ["t2v_i2v", "r2v", "both"]
DIT_QUANT = "int8"  #@param ["int8", "fp8"]
BRANCH = "cursor/h3-live-remake-on-minimax-80fd"
URL = f"https://raw.githubusercontent.com/fireworker011/Research/{BRANCH}/h3/drive_models.py"

if not os.path.isdir(DRIVE_MODELS):
    raise SystemExit("先にセル1で Drive をマウントする。models が無い: " + DRIVE_MODELS)
if not os.path.isdir(COMFY_DIR):
    raise SystemExit("先にセル2で ComfyUI を置く。")

module_path = Path("/content/drive_models.py")
subprocess.run(["wget", "-q", "-O", str(module_path), URL], check=True)
spec = importlib.util.spec_from_file_location("drive_models", module_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
try:
    module.require_mode(Path(DRIVE_MODELS), MODE, DIT_QUANT)
    linked = module.link_comfy_models(Path(COMFY_DIR) / "models", Path(DRIVE_MODELS))
except module.DriveModelsError as exc:
    raise SystemExit(str(exc)) from exc
print("linked", ", ".join(linked))
print("動画 VAE は Drive にある fp16。int8 は落とさない。")
print("生成ボタンは押していない。")

In [ ]:
#@title 4) 公式 MiniMax H3 workflow を配置
import os

COMFY_DIR = "/content/ComfyUI"
wf_dir = f"{COMFY_DIR}/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)

base = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates"
for name in [
    "video_minimax_h3_t2v.json",
    "video_minimax_h3_i2v.json",
    "video_minimax_h3_r2v.json",
]:
    !wget -q -O "{wf_dir}/{name}" "{base}/{name}"
    print("saved:", name)

print("UI で Workflows から読み込むか、JSON をドラッグ＆ドロップ")


In [ ]:
#@title 5) ComfyUI 起動 + Colab プロキシ URL（trycloudflare 不使用）
import os
import signal
import subprocess
import sys
import time
import urllib.request

COMFY_DIR = "/content/ComfyUI"
PORT = 8188
# T4: --lowvram / さらに厳しい: --novram
# L4・A100: --normalvram または --highvram
EXTRA_ARGS = "--normalvram"  #@param ["--highvram", "--normalvram", "--lowvram", "--novram"]
READY_TIMEOUT = 180
LOG_PATH = "/content/comfyui.log"


def kill_old():
    for c in [
        f"fuser -k {PORT}/tcp",
        "pkill -f 'python.*main.py'",
        "pkill -f cloudflared",
        "pkill -f localtunnel",
    ]:
        subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)


def wait_ready(proc, timeout):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
            return True
        except Exception:
            time.sleep(2)
    return False


def log_tail(n=100):
    try:
        with open(LOG_PATH, "r", errors="replace") as f:
            print("".join(f.readlines()[-n:]))
    except FileNotFoundError:
        print("(no log)")


if not os.path.isfile(os.path.join(COMFY_DIR, "main.py")):
    raise SystemExit(f"ComfyUI がありません: {COMFY_DIR}  — 先にセル2を実行")

os.chdir(COMFY_DIR)
kill_old()

log_f = open(LOG_PATH, "w", buffering=1)
cmd = [
    sys.executable, "main.py",
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    EXTRA_ARGS,
    "--disable-auto-launch",
]
print("starting:", " ".join(cmd))
print("log:", LOG_PATH)

proc = subprocess.Popen(
    cmd,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"起動待ち（最大 {READY_TIMEOUT}s）...")
if not wait_ready(proc, READY_TIMEOUT):
    print("起動失敗。ログ末尾:")
    log_f.flush()
    log_tail()
    raise SystemExit(1)

print(f"OK: ComfyUI listening on 0.0.0.0:{PORT}")

# --- Colab 公式プロキシ（本命）---
url = None
try:
    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
except Exception as e:
    print("proxyPort 失敗:", e)

print()
print("=" * 64)
print("  この URL を開いてください（Colab プロキシ）")
print("=" * 64)
print(url if url else "(URL 取得失敗 → セル5b フォールバックを実行)")
print("=" * 64)
print("  ※ trycloudflare.com は使わない")
print("=" * 64)
print()

try:
    from google.colab import output as colab_output
    colab_output.serve_kernel_port_as_window(PORT)
    print("（ポップアップ許可時は別タブも開きます）")
except Exception as e:
    print("window open:", e)

print("ComfyUI は起動したままにする。次のセルが span を1本入れる。")
print("このセルを止めてもサーバーは落ちない。")
globals()["_COMFY_PROC"] = proc
globals()["_COMFY_LOG"] = log_f


In [ ]:
#@title 5b) span を1本だけ投入（既定は span-01）
PLAN = ""  #@param {type:"string"}
SHEET = "/content/drive/MyDrive/動画倉庫/ネタBOX/1024-sheet.json"  #@param {type:"string"}
SPAN = 1  #@param {type:"integer"}
OUT_DIR = "/content/drive/MyDrive/動画倉庫/00_確認待ち/2026-10-26_H3カイロ_テスト"  #@param {type:"string"}
SEED = 1024  #@param {type:"integer"}
PORT = 8188

import importlib
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from google.colab import drive

print("このセルは Comfy の /prompt に1本入れる。UI の生成ボタンは押さない。")

if not os.path.isdir("/content/drive/MyDrive"):
    print("Drive の接続画面では ys73497349@gmail.com を選ぶ。")
    drive.mount("/content/drive")

BRANCH = "cursor/h3-live-remake-on-minimax-80fd"
RAW = f"https://raw.githubusercontent.com/fireworker011/Research/{BRANCH}/h3"
pkg = Path("/content/h3pkg")
pkg.mkdir(exist_ok=True)
(pkg / "__init__.py").write_text("", encoding="utf-8")
for name in ("live_remake.py", "span_queue.py"):
    subprocess.run(["wget", "-q", "-O", str(pkg / name), f"{RAW}/{name}"], check=True)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
importlib.invalidate_caches()
for mod in ("h3pkg.live_remake", "h3pkg.span_queue"):
    sys.modules.pop(mod, None)
from h3pkg.live_remake import build_remake_plan, load_sheet
from h3pkg.span_queue import (
    build_ref2va_prompt,
    finished_history,
    history_video,
    mux_source_audio_argv,
    output_paths,
    post_prompt,
    select_span,
    video_vae_name,
)

if PLAN:
    plan = json.loads(Path(PLAN).read_text(encoding="utf-8"))
else:
    if not SHEET:
        raise SystemExit("PLAN か SHEET のパスを入れる。")
    plan = build_remake_plan(load_sheet(SHEET))
span = select_span(plan, int(SPAN))
os.makedirs(OUT_DIR, exist_ok=True)
video_out, audio_out = output_paths(OUT_DIR, int(SPAN))

base = f"http://127.0.0.1:{PORT}"
try:
    urllib.request.urlopen(base + "/system_stats", timeout=5)
except Exception as exc:
    raise SystemExit(f"ComfyUI が {PORT} で動いていない。先に起動セルを実行する。{exc}") from exc

comfy = Path("/content/ComfyUI")
vae_dir = comfy / "models" / "vae"
present = [item.name for item in vae_dir.glob("*.safetensors")] if vae_dir.is_dir() else []
vae = video_vae_name(plan, present)
if not (vae_dir / vae).is_file():
    raise SystemExit(f"動画 VAE が Drive の models/vae に無い: {vae}。落とさない。")

input_dir = comfy / "input"
input_dir.mkdir(parents=True, exist_ok=True)
image_files = []
for picture in span["pictures"]:
    src = Path(picture["path"])
    if not src.is_file():
        raise SystemExit(f"静止画が無い: {src}")
    shutil.copy2(src, input_dir / src.name)
    image_files.append(src.name)
    print("ref", picture["label"], src.name)

ref_path = input_dir / f"span-{int(SPAN):02d}-ref.mp4"
argv = list(span["conform_argv"])
argv[-1] = str(ref_path)
subprocess.run(argv, check=True)
counted = subprocess.check_output(
    [
        "ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
        "-show_entries", "stream=nb_read_frames", "-of", "csv=p=0", str(ref_path),
    ],
    text=True,
).strip()
print("参照動画のコマ", counted, "要求", span["length"])
if counted != str(span["length"]):
    raise SystemExit(f"参照動画が {span['length']} コマではない: {counted}")

prompt = build_ref2va_prompt(
    plan,
    span,
    image_files=image_files,
    video_file=ref_path.name,
    video_vae=vae,
    seed=int(SEED),
)
prompt_id = post_prompt(base, prompt)
print("prompt_id", prompt_id, "steps", plan["model"]["steps"], "length", span["length"])

deadline = time.time() + 60 * 60
history = None
while time.time() < deadline:
    try:
        with urllib.request.urlopen(f"{base}/history/{prompt_id}", timeout=30) as response:
            body = json.loads(response.read().decode("utf-8"))
    except Exception:
        time.sleep(15)
        continue
    history = finished_history(body, prompt_id)
    if history is not None:
        break
    time.sleep(15)
if history is None:
    raise SystemExit("1時間待っても完了しない。/content/comfyui.log を見る。")

saved = history_video(history)
if not saved:
    raise SystemExit("履歴に mp4 が無い。")
found = list((comfy / "output").rglob(saved))
if not found:
    raise SystemExit(f"出力ファイルが無い: {saved}")
video_out, audio_out = output_paths(OUT_DIR, int(SPAN))
shutil.copy2(found[0], video_out)
subprocess.run(
    mux_source_audio_argv(str(video_out), plan["inputs"]["video"], float(span["start_s"]), int(span["frames"]), str(audio_out)),
    check=True,
)
print("WROTE", video_out)
print("WROTE", audio_out)
print("生成ボタンは押していない。")

In [ ]:
import urllib.request
from google.colab import output as colab_output

# 生きてるか
urllib.request.urlopen(f"http://127 001: {PORT}/", timeout=3)
print("ComfyUI OK")

#
colab_output. serve_kernel_port_as_iframe(PORT, height=920)
# 2外部用(loca.lt)
!curl -s https://loca.lt/mytunnelpassword
print("←初回パスワード(IP)")
! npx -- yes localtunnel -- port 8188

PORT = 8188

In [ ]:
#@title 5b) フォールバック: localtunnel（セル5が動いている状態で実行）
# セル5の起動ループと同時には回せない場合あり。
# そのときはセル5を実行したまま、別ランタイムではなく
# 起動を background 化したうえで本セルを使うか、
# 下の「バックグラウンド起動版」を使う。

PORT = 8188
import shutil
import subprocess
import urllib.request

if shutil.which("npm") is None:
    print("Installing Node.js...")
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
    !apt-get install -y -qq nodejs

print("localtunnel 初回パスワード（IP）:")
try:
    print(urllib.request.urlopen("https://loca.lt/mytunnelpassword", timeout=10).read().decode().strip())
except Exception as e:
    print(e)
    !curl -s https://loca.lt/mytunnelpassword || true

print()
print("表示された https://xxxx.loca.lt を開く（trycloudflare ではない）")
!npx --yes localtunnel --port {PORT}


In [ ]:
#@title 6) (任意) 生成結果を Drive にコピー
USE_DRIVE = False  #@param {type:"boolean"}
DRIVE_OUT = "/content/drive/MyDrive/minimax-h3-comfyui/outputs"  #@param {type:"string"}

if USE_DRIVE:
    import os
    import shutil
    src = "/content/ComfyUI/output"
    os.makedirs(DRIVE_OUT, exist_ok=True)
    if os.path.isdir(src):
        n = 0
        for root, _, files in os.walk(src):
            for f in files:
                if f.lower().endswith((".mp4", ".png", ".webm", ".gif", ".webp")):
                    shutil.copy2(os.path.join(root, f), os.path.join(DRIVE_OUT, f))
                    print("copied:", f)
                    n += 1
        print("total:", n)
    else:
        print("output がまだありません")
else:
    print("USE_DRIVE=True にして再実行")


## UI を開いたら

1. **Template Library → Video → MiniMax H3**（T2V / I2V / R2V）
2. モデル確認:
   - DiT: `minimax_h3_fl2va_pruned_int8_convrot.safetensors`（T2V/I2V）
   - TE: `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`
   - VAE: video + audio
3. T4 なら秒数短め・解像度低めから
4. 生成物は `ComfyUI/output`（必要ならセル6で Drive へ）

公式: https://docs.comfy.org/tutorials/video/minimax/minimax-h3
